In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM, BigBirdModel, BigBirdTokenizer
import ijson
import random
import re
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score, log_loss
import xgboost as xgb
from scipy import sparse


In [ ]:
# load Data/data_augmented.json file into a dictionary
with open('Data/train_data.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)

with open('Data/val_data.json', 'r', encoding='utf-8') as f:
    val_data = json.load(f)

with open('Data/test_data.json', 'r', encoding='utf-8') as f:
    test_data = json.load(f)

In [ ]:
train_df = pd.DataFrame(train_data)
train_df["min_age"] = train_df["ages"].apply(lambda x: min(x) if x else None)
train_df["max_age"] = train_df["ages"].apply(lambda x: max(x) if x else None)
train_df["median_age"] = train_df["ages"].apply(lambda x: np.median(x) if x else None)

val_df = pd.DataFrame(val_data)
val_df["min_age"] = val_df["ages"].apply(lambda x: min(x) if x else None)
val_df["max_age"] = val_df["ages"].apply(lambda x: max(x) if x else None)
val_df["median_age"] = val_df["ages"].apply(lambda x: np.median(x) if x else None)

test_df = pd.DataFrame(test_data)
test_df["min_age"] = test_df["ages"].apply(lambda x: min(x) if x else None)
test_df["max_age"] = test_df["ages"].apply(lambda x: max(x) if x else None)
test_df["median_age"] = test_df["ages"].apply(lambda x: np.median(x) if x else None)


In [ ]:
# import matplotlib.pyplot as plt

# # flatten all statutes (keep duplicates to count frequency)
# statutes = [s for item in data for s in item["statute_details"]]

# tokenizer = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")
# encoded_batch = tokenizer(
#     statutes,
#     add_special_tokens=True,
#     return_attention_mask=False,
#     return_token_type_ids=False,
# )

# token_counts = [len(ids) for ids in encoded_batch["input_ids"]]

# # frequency of token counts
# freq = pd.Series(token_counts).value_counts().sort_index()

# # bucket token counts in steps of 100 for the bar graph
# bins = np.arange(0, max(token_counts) + 100, 100)
# labels = [f"{b}-{b+99}" for b in bins[:-1]]
# freq = pd.Series(pd.cut(token_counts, bins=bins, right=False, labels=labels)).value_counts().sort_index()

# freq.plot(kind="bar", figsize=(12, 4))
# plt.xlabel("Token count (bucket size = 100)")
# plt.ylabel("Frequency")
# plt.title("Token count distribution for statutes (bucketed)")
# plt.tight_layout()
# plt.show()

# print(f"Example: {statutes[0][:80]}... -> {token_counts[0]} tokens")

In [ ]:
len(train_df), len(val_df), len(test_df)

In [ ]:
# make a train_statutes_df from train_df which will have the CNR field, and statutes field. statutes field will be a merged list train_df['statutes'] + 'past_crimes_present' if train_df['past_criminal_records_exists'] is True
# + "past__" added as prefix to each item in train_df['past_criminal_record_charges'] if its not None.
# it will also have a outcome field which will be 0 if train_df['outcome'] is 'bail not granted' else 1 if train_df['outcome'] is 'bail granted'
train_statutes_df = pd.DataFrame()
train_statutes_df['CNR'] = train_df['CNR']
train_statutes_df['statutes'] = train_df.apply(lambda row: row['statutes'] + (['past_crimes_present'] if row['past_criminal_record_exists'] else []) + (['past__' + charge for charge in row['past_criminal_record_charges']] if row['past_criminal_record_charges'] else []), axis=1)
train_statutes_df['outcome'] = train_df['outcome'].apply(lambda x: 0 if x == 'bail not granted' else 1 if x == 'bail granted' else None)
train_statutes_df = train_statutes_df.reset_index(drop=True)


In [ ]:
val_statutes_df = pd.DataFrame()
val_statutes_df['CNR'] = val_df['CNR']
val_statutes_df['statutes'] = val_df.apply(lambda row: row['statutes'] + (['past_crimes_present'] if row['past_criminal_record_exists'] else []) + (['past__' + charge for charge in row['past_criminal_record_charges']] if row['past_criminal_record_charges'] else []), axis=1)
val_statutes_df['outcome'] = val_df['outcome'].apply(lambda x: 0 if x == 'bail not granted' else 1 if x == 'bail granted' else None)
val_statutes_df = val_statutes_df.reset_index(drop=True)

test_statutes_df = pd.DataFrame()
test_statutes_df['CNR'] = test_df['CNR']
test_statutes_df['statutes'] = test_df.apply(lambda row: row['statutes'] + (['past_crimes_present'] if row['past_criminal_record_exists'] else []) + (['past__' + charge for charge in row['past_criminal_record_charges']] if row['past_criminal_record_charges'] else []), axis=1)
test_statutes_df['outcome'] = test_df['outcome'].apply(lambda x: 0 if x == 'bail not granted' else 1 if x == 'bail granted' else None)
test_statutes_df = test_statutes_df.reset_index(drop=True)

In [ ]:
# count how many rows have statutes as empty list in train_statutes_df, val_statutes_df and test_statutes_df
(len(train_statutes_df[train_statutes_df['statutes'].apply(lambda x: len(x) == 0)]),
len(val_statutes_df[val_statutes_df['statutes'].apply(lambda x: len(x) == 0)]),
len(test_statutes_df[test_statutes_df['statutes'].apply(lambda x: len(x) == 0)]))

# # save the CNR of rows which have empty statutes in train_statutes_df, val_statutes_df and test_statutes_df to a text file
# with open('Data/empty_statutes_cnrs.txt', 'w') as f:
#     # f.write("Train set empty statutes CNRs:\n")
#     for cnr in train_statutes_df[train_statutes_df['statutes'].apply(lambda x: len(x) == 0)]['CNR']:
#         f.write(cnr + "\n")
#     # f.write("\nValidation set empty statutes CNRs:\n")
#     for cnr in val_statutes_df[val_statutes_df['statutes'].apply(lambda x: len(x) == 0)]['CNR']:
#         f.write(cnr + "\n")
#     # f.write("\nTest set empty statutes CNRs:\n")
#     for cnr in test_statutes_df[test_statutes_df['statutes'].apply(lambda x: len(x) == 0)]['CNR']:
#         f.write(cnr + "\n")

In [ ]:
mlb = MultiLabelBinarizer(sparse_output=True)
X = mlb.fit_transform(train_statutes_df["statutes"])          # scipy sparse matrix (N, V)
y = train_statutes_df["outcome"].astype(int).to_numpy()       # (N,)
feature_names = mlb.classes_

In [ ]:
def shap_summary_8(shap_vals: np.ndarray) -> np.ndarray:
    """
    shap_vals: (n_samples, n_features)
    returns:   (n_samples, 8)
    """
    pos = np.where(shap_vals > 0, shap_vals, 0.0)
    neg = np.where(shap_vals < 0, shap_vals, 0.0)

    sum_pos = pos.sum(axis=1)
    sum_neg = neg.sum(axis=1)  # negative number

    max_pos = pos.max(axis=1)
    min_neg = neg.min(axis=1)  # most negative

    pos_count = (shap_vals > 0).sum(axis=1).astype(np.float32)
    neg_count = (shap_vals < 0).sum(axis=1).astype(np.float32)

    abs_vals = np.abs(shap_vals)
    l1_total = abs_vals.sum(axis=1)

    # top-3 abs sum (fast, no full sort)
    k = 3
    if shap_vals.shape[1] >= k:
        topk = np.partition(abs_vals, -k, axis=1)[:, -k:]
        top3_abs_sum = topk.sum(axis=1)
    else:
        top3_abs_sum = abs_vals.sum(axis=1)

    out = np.vstack([sum_pos, sum_neg, max_pos, min_neg,
                     pos_count, neg_count, l1_total, top3_abs_sum]).T
    return out.astype(np.float32)


In [ ]:

X_train = mlb.fit_transform(train_statutes_df["statutes"])
y_train = train_statutes_df["outcome"].astype(int).to_numpy()

X_val = mlb.transform(val_statutes_df["statutes"])
y_val = val_statutes_df["outcome"].astype(int).to_numpy()


best = None

param_grid = [
    dict(max_depth=5, min_child_weight=12, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0),
    dict(max_depth=5, min_child_weight=12, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0),
    dict(max_depth=5, min_child_weight=12, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0),
    dict(max_depth=6, min_child_weight=12, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0),
]

for p in param_grid:
    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        learning_rate=0.05,
        n_estimators=5000,              # large; early stopping will decide
        tree_method="hist",
        n_jobs=-1,
        device='cuda',
        early_stopping_rounds=100,
        **p
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    val_prob = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_prob)
    ll  = log_loss(y_val, val_prob)

    score = ll  # choose logloss as primary

    print(p, "best_iter=", model.best_iteration, "logloss=", ll, "auc=", auc)

    if best is None or score < best["score"]:
        best = {"params": p, "score": score, "auc": auc, "logloss": ll, "best_iter": model.best_iteration}

print("BEST:", best)


In [ ]:
K = 10
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)

# Preallocate OOF outputs
oof_shap8 = np.zeros((X.shape[0], 8), dtype=np.float32)
oof_pred  = np.zeros((X.shape[0],), dtype=np.float32)

xgb_params = dict(
    n_estimators=800,
    early_stopping_rounds=100, 
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_weight=12,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",      
    device="cuda",
    n_jobs=-1,
)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, y_tr = X[tr_idx], y[tr_idx]
    X_va, y_va = X[va_idx], y[va_idx]

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )

    # OOF predicted probability
    oof_pred[va_idx] = model.predict_proba(X_va)[:, 1].astype(np.float32)

    # ---- TreeSHAP via pred_contribs ----
    booster = model.get_booster()

    dval = xgb.DMatrix(X_va)
    contribs = booster.predict(dval, pred_contribs=True)
    # contribs shape: (n_val, n_features + 1)
    # last column = bias (base value)

    shap_vals = contribs[:, :-1]   # drop bias

    # summarize into 8 SHAP features
    oof_shap8[va_idx] = shap_summary_8(shap_vals)

    print(f"Fold {fold}/{K} done. val size={len(va_idx)}")


In [ ]:
shap_cols = [
    "shap_sum_pos", "shap_sum_neg", "shap_max_pos", "shap_min_neg",
    "shap_pos_count", "shap_neg_count", "shap_l1_total", "shap_top3_abs_sum"
]

for j, c in enumerate(shap_cols):
    train_statutes_df[c] = oof_shap8[:, j]

train_statutes_df["xgb_oof_pred"] = oof_pred


In [ ]:
# No missing OOF rows
assert np.isfinite(train_statutes_df[shap_cols].to_numpy()).all()
assert np.isfinite(train_statutes_df["xgb_oof_pred"].to_numpy()).all()

# Basic check: OOF pred should correlate with outcome (not perfect)
print(train_statutes_df.groupby("outcome")["xgb_oof_pred"].mean())

In [ ]:
X_train_full = mlb.transform(train_statutes_df["statutes"])
y_train_full = train_statutes_df["outcome"].astype(int).to_numpy()

X_val = mlb.transform(val_statutes_df["statutes"])
y_val = val_statutes_df["outcome"].astype(int).to_numpy()

X_test = mlb.transform(test_statutes_df["statutes"])
y_test = test_statutes_df["outcome"].astype(int).to_numpy()


In [ ]:
final_xgb = xgb.XGBClassifier(**xgb_params)

final_xgb.fit(
    X_train_full,
    y_train_full,
    eval_set=[(X_val, y_val)],   # optional: just for monitoring
    verbose=False
)


In [ ]:
# predict on val and get accuracy, logloss, auc
val_prob = final_xgb.predict_proba(X_val)[:, 1]
val_pred = (val_prob >= 0.5).astype(int)
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
accuracy = accuracy_score(y_val, val_pred)
logloss = log_loss(y_val, val_prob)
auc = roc_auc_score(y_val, val_prob)
print(f"Final XGB on Val -> Accuracy: {accuracy:.4f}, LogLoss: {logloss:.4f}, AUC: {auc:.4f}")

In [ ]:
val_xgb_pred  = final_xgb.predict_proba(X_val)[:, 1].astype(np.float32)
test_xgb_pred = final_xgb.predict_proba(X_test)[:, 1].astype(np.float32)

# ---- TreeSHAP via pred_contribs=True (no densifying needed) ----
booster = final_xgb.get_booster()

dval  = xgb.DMatrix(X_val)
dtest = xgb.DMatrix(X_test)

val_contrib  = booster.predict(dval,  pred_contribs=True)   # (n_val, n_feat+1)
test_contrib = booster.predict(dtest, pred_contribs=True)   # (n_test, n_feat+1)

# Drop bias column (last column)
shap_val  = val_contrib[:, :-1]
shap_test = test_contrib[:, :-1]

# Summarize to 8 features
val_shap8  = shap_summary_8(shap_val)
test_shap8 = shap_summary_8(shap_test)

# Store back in dataframes
for j, c in enumerate(shap_cols):
    val_statutes_df[c] = val_shap8[:, j]
val_statutes_df["xgb_pred"] = val_xgb_pred

for j, c in enumerate(shap_cols):
    test_statutes_df[c] = test_shap8[:, j]
test_statutes_df["xgb_pred"] = test_xgb_pred

In [ ]:
# Check shapes
assert train_statutes_df[shap_cols].shape[1] == 8
assert val_statutes_df[shap_cols].shape[1] == 8
assert test_statutes_df[shap_cols].shape[1] == 8

# Check finite
assert np.isfinite(val_statutes_df[shap_cols].to_numpy()).all()
assert np.isfinite(test_statutes_df[shap_cols].to_numpy()).all()

# Check signal
print(val_statutes_df.groupby("outcome")["xgb_pred"].mean())


In [ ]:
train_statutes_df.head()

In [ ]:
train_df.head()

In [ ]:
# make a new dataframe final_train_df with following specifications:
# 1. it will be a merged dataframe of train_df and train_statutes_df on CNR field
# 2. it will have CNR field
# 3. bail_type field will be 1 if train_df['bail_type'] is 'regular bail' else 2
# 4. age_available field will be 1 if train_df['age_available'] is True else 0
# 5. there will be a details field which will have train_df['case_details'] + ('health conditions faced by the accused are ' + train_df['health_conditions'] if train_df['health_conditions_available'] is True else '')
# 6. all the fields of train_statutes_df except CNR and statutes field (include the outcome field)
# 7. days_in_custody field from train_df (make it None if missing or NaN)
# 8. min_age, max_age, median_age fields from train_df (make it None if missing or NaN)


# 1) Merge on CNR, keep only required columns from train_statutes_df
stat_cols = [
    "CNR",
    "outcome",
    "shap_sum_pos", "shap_sum_neg", "shap_max_pos", "shap_min_neg",
    "shap_pos_count", "shap_neg_count", "shap_l1_total", "shap_top3_abs_sum",
    "xgb_oof_pred",
]
final_train_df = train_df.merge(train_statutes_df[stat_cols], on="CNR", how="inner")

# 2) CNR already present from merge

# 3) bail_type: 1 if 'regular bail' else 2
final_train_df["bail_type"] = np.where(
    final_train_df["bail_type"].astype(str).str.strip().str.lower().eq("regular-bail"),
    1, 2
)

# 4) age_available: 1 if True else 0
final_train_df["age_available"] = final_train_df["age_available"].fillna(False).astype(int)

# 5) details: case_details + optional health text
def _clean_str(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    return s if s else None

def build_details(row):
    base = _clean_str(row.get("case_details")) or ""
    hc = _clean_str(row.get("health_condition"))
    # include health condition only if not None and not "none."
    if hc is not None and hc.strip().lower() != "none." and hc.strip().lower() != "none":
        suffix = f" health conditions faced by the accused are {hc}"
        return (base + suffix).strip()
    return base.strip() if base.strip() else None

final_train_df["details"] = final_train_df.apply(build_details, axis=1)

# 7) days_in_custody_available: 0 if missing/NaN else 1
final_train_df["days_in_custody_available"] = np.where(
    final_train_df["days_in_custody"].isna(), 0, 1
).astype(int)

# 8) days_in_custody: keep value, but normalize NaN -> None
final_train_df["days_in_custody"] = final_train_df["days_in_custody"].where(
    ~final_train_df["days_in_custody"].isna(), None
)

# 9) min_age, max_age, median_age: normalize NaN -> None
for c in ["min_age", "max_age", "median_age"]:
    final_train_df[c] = final_train_df[c].where(~final_train_df[c].isna(), None)

# Final column order (exactly as requested)
final_cols = [
    "CNR",
    "bail_type",
    "details",
    "days_in_custody_available",
    "days_in_custody",
    "age_available",
    "min_age",
    "max_age",
    "median_age",
    "shap_sum_pos",
    "shap_sum_neg",
    "shap_max_pos",
    "shap_min_neg",
    "shap_pos_count",
    "shap_neg_count",
    "shap_l1_total",
    "shap_top3_abs_sum",
    "xgb_oof_pred",
    "outcome_y",
]

final_train_df = final_train_df[final_cols]
final_train_df.rename(columns={"outcome_y": "outcome"}, inplace=True)


In [ ]:
final_train_df.columns

In [ ]:
# 1) Merge on CNR, keep only required columns from train_statutes_df
stat_cols = [
    "CNR",
    "outcome",
    "shap_sum_pos", "shap_sum_neg", "shap_max_pos", "shap_min_neg",
    "shap_pos_count", "shap_neg_count", "shap_l1_total", "shap_top3_abs_sum",
    "xgb_pred",
]
final_val_df = val_df.merge(val_statutes_df[stat_cols], on="CNR", how="inner")

# 2) CNR already present from merge

# 3) bail_type: 1 if 'regular bail' else 2
final_val_df["bail_type"] = np.where(
    final_val_df["bail_type"].astype(str).str.strip().str.lower().eq("regular-bail"),
    1, 2
)

# 4) age_available: 1 if True else 0
final_val_df["age_available"] = final_val_df["age_available"].fillna(False).astype(int)

# 5) details: case_details + optional health text
def _clean_str(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    return s if s else None

def build_details(row):
    base = _clean_str(row.get("case_details")) or ""
    hc = _clean_str(row.get("health_condition"))
    # include health condition only if not None and not "none."
    if hc is not None and hc.strip().lower() != "none." and hc.strip().lower() != "none":
        suffix = f" health conditions faced by the accused are {hc}"
        return (base + suffix).strip()
    return base.strip() if base.strip() else None

final_val_df["details"] = final_val_df.apply(build_details, axis=1)

# 7) days_in_custody_available: 0 if missing/NaN else 1
final_val_df["days_in_custody_available"] = np.where(
    final_val_df["days_in_custody"].isna(), 0, 1
).astype(int)

# 8) days_in_custody: keep value, but normalize NaN -> None
final_val_df["days_in_custody"] = final_val_df["days_in_custody"].where(
    ~final_val_df["days_in_custody"].isna(), None
)

# 9) min_age, max_age, median_age: normalize NaN -> None
for c in ["min_age", "max_age", "median_age"]:
    final_val_df[c] = final_val_df[c].where(~final_val_df[c].isna(), None)

# Final column order (exactly as requested)
final_cols = [
    "CNR",
    "bail_type",
    "details",
    "days_in_custody_available",
    "days_in_custody",
    "age_available",
    "min_age",
    "max_age",
    "median_age",
    "shap_sum_pos",
    "shap_sum_neg",
    "shap_max_pos",
    "shap_min_neg",
    "shap_pos_count",
    "shap_neg_count",
    "shap_l1_total",
    "shap_top3_abs_sum",
    "xgb_pred",
    "outcome_y",
]

final_val_df = final_val_df[final_cols]
final_val_df.rename(columns={"outcome_y": "outcome"}, inplace=True)

In [ ]:
# 1) Merge on CNR, keep only required columns from train_statutes_df
stat_cols = [
    "CNR",
    "outcome",
    "shap_sum_pos", "shap_sum_neg", "shap_max_pos", "shap_min_neg",
    "shap_pos_count", "shap_neg_count", "shap_l1_total", "shap_top3_abs_sum",
    "xgb_pred",
]
final_test_df = test_df.merge(test_statutes_df[stat_cols], on="CNR", how="inner")

# 2) CNR already present from merge

# 3) bail_type: 1 if 'regular bail' else 2
final_test_df["bail_type"] = np.where(
    final_test_df["bail_type"].astype(str).str.strip().str.lower().eq("regular-bail"),
    1, 2
)

# 4) age_available: 1 if True else 0
final_test_df["age_available"] = final_test_df["age_available"].fillna(False).astype(int)

# 5) details: case_details + optional health text
def _clean_str(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    return s if s else None

def build_details(row):
    base = _clean_str(row.get("case_details")) or ""
    hc = _clean_str(row.get("health_condition"))
    # include health condition only if not None and not "none."
    if hc is not None and hc.strip().lower() != "none." and hc.strip().lower() != "none":
        suffix = f" health conditions faced by the accused are {hc}"
        return (base + suffix).strip()
    return base.strip() if base.strip() else None

final_test_df["details"] = final_test_df.apply(build_details, axis=1)

# 7) days_in_custody_available: 0 if missing/NaN else 1
final_test_df["days_in_custody_available"] = np.where(
    final_test_df["days_in_custody"].isna(), 0, 1
).astype(int)

# 8) days_in_custody: keep value, but normalize NaN -> None
final_test_df["days_in_custody"] = final_test_df["days_in_custody"].where(
    ~final_test_df["days_in_custody"].isna(), None
)

# 9) min_age, max_age, median_age: normalize NaN -> None
for c in ["min_age", "max_age", "median_age"]:
    final_test_df[c] = final_test_df[c].where(~final_test_df[c].isna(), None)

# Final column order (exactly as requested)
final_cols = [
    "CNR",
    "bail_type",
    "details",
    "days_in_custody_available",
    "days_in_custody",
    "age_available",
    "min_age",
    "max_age",
    "median_age",
    "shap_sum_pos",
    "shap_sum_neg",
    "shap_max_pos",
    "shap_min_neg",
    "shap_pos_count",
    "shap_neg_count",
    "shap_l1_total",
    "shap_top3_abs_sum",
    "xgb_pred",
    "outcome_y",
]

final_test_df = final_test_df[final_cols]
final_test_df.rename(columns={"outcome_y": "outcome"}, inplace=True)

In [ ]:
final_test_df.head()

In [ ]:
final_train_df.to_pickle("Data/final_train_df.pkl")
final_val_df.to_pickle("Data/final_val_df.pkl")
final_test_df.to_pickle("Data/final_test_df.pkl")

In [ ]:
final_train_df.info()